# Fleiss' Kappa — 4 位标注员一致性计算

**数据**: `Data/Human.csv`（仅使用 4 位标注员的原始标注列，忽略综合列与 LLM 列）

| 列 | 含义 |
|---|---|
| `负面性1` ~ `负面性4` | 标注员 1~4 对「负面性」的二分类标注 (0/1) |
| `侮辱性1` ~ `侮辱性4` | 标注员 1~4 对「侮辱性」的二分类标注 (0/1) |

**Fleiss' Kappa 定义**（多标注员、多类别）:

$$P_i = \frac{\sum_j n_{ij}^2 - n}{n(n-1)}, \quad \bar{P} = \frac{1}{N}\sum_i P_i$$

$$p_j = \frac{\sum_i n_{ij}}{N \cdot n}, \quad P_e = \sum_j p_j^2$$

$$\kappa = \frac{\bar{P} - P_e}{1 - P_e}$$

其中 $N$=样本数, $n$=每条样本的标注员数(=4), $k$=类别数(=2), $n_{ij}$=第 $i$ 条被标为类别 $j$ 的票数。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../Data/Human.csv")
DIMS = ["负面性", "侮辱性"]   # 待计算的两个标注维度
N_RATERS = 4                  # 每条样本的标注员数量

In [2]:
df = pd.read_csv(DATA_PATH)

# 数据完整性检查
required = [f"{d}{i}" for d in DIMS for i in range(1, N_RATERS + 1)]
missing = [c for c in required if c not in df.columns]
assert not missing, f"缺少标注列: {missing}"

print(f"样本数 N = {len(df)}，标注员 n = {N_RATERS}，缺失标注 = {df[required].isna().sum().sum()}")

样本数 N = 423，标注员 n = 4，缺失标注 = 0


In [3]:
def fleiss_kappa(mat):
    """计算 Fleiss' Kappa。

    Parameters
    ----------
    mat : np.ndarray, shape (N, k)
        票数矩阵：mat[i, j] = 第 i 条样本被标为类别 j 的标注员票数。
        每行票数之和必须等于 n（标注员数）。

    Returns
    -------
    (kappa, P_bar, P_e)
        kappa : Fleiss' Kappa；若 1 - P_e == 0（数据无任何变化）则返回 None
        P_bar : 总观察一致率
        P_e   : 期望一致率
    """
    N, k = mat.shape
    n = mat.sum(axis=1)[0]
    assert (mat.sum(axis=1) == n).all(), "每行票数不一致：标注员数量不齐"

    p_j = mat.sum(axis=0) / (N * n)                      # 类别 j 的整体比例
    P_i = (np.square(mat).sum(axis=1) - n) / (n * (n - 1))  # 每条的观察一致程度
    P_bar, P_e = P_i.mean(), np.square(p_j).sum()

    if np.isclose(1 - P_e, 0):
        return None, P_bar, P_e   # 退化情形：全员一致且只出现一个类别
    return (P_bar - P_e) / (1 - P_e), P_bar, P_e


def build_vote_matrix(df, dim):
    """把某维度的 4 列标注转成 N x 2 票数矩阵（类别 0 / 类别 1）。"""
    mat = np.zeros((len(df), 2), dtype=int)
    for i in range(1, N_RATERS + 1):
        labels = df[f"{dim}{i}"].astype(int).to_numpy()
        mat[np.arange(len(df)), labels] += 1
    return mat

In [4]:
results = {}
for dim in DIMS:
    mat = build_vote_matrix(df, dim)
    kappa, P_bar, P_e = fleiss_kappa(mat)
    results[dim] = kappa

    ones = mat[:, 1]   # 每条样本被标为 1 的票数
    n_disagree = int(((ones > 0) & (ones < N_RATERS)).sum())
    full_agree = 1 - n_disagree / len(df)

    print(f"===== {dim} =====")
    print(f"  观察一致率 P_bar = {P_bar:.4f}")
    print(f"  期望一致率 P_e   = {P_e:.4f}")
    kappa_str = f"{kappa:.4f}" if kappa is not None else "未定义 (1 - P_e = 0)"
    print(f"  Fleiss' Kappa    = {kappa_str}")
    print(f"  4人完全一致 = {full_agree:.2%}，存在分歧 = {n_disagree} 条")
    dist = pd.Series(ones).value_counts().sort_index().to_dict()
    print(f"  标1人数分布 (0~4人): {dist}")
    print()

===== 负面性 =====
  观察一致率 P_bar = 0.9799
  期望一致率 P_e   = 0.5053
  Fleiss' Kappa    = 0.9594
  4人完全一致 = 96.45%，存在分歧 = 15 条
  标1人数分布 (0~4人): {0: 183, 1: 3, 2: 6, 3: 6, 4: 225}

===== 侮辱性 =====
  观察一致率 P_bar = 0.9752
  期望一致率 P_e   = 0.8240
  Fleiss' Kappa    = 0.8590
  4人完全一致 = 95.51%，存在分歧 = 19 条
  标1人数分布 (0~4人): {0: 372, 1: 7, 2: 6, 3: 6, 4: 32}



In [5]:
# 汇总表
summary = pd.DataFrame(
    [{"维度": dim, "Fleiss' Kappa": kappa} for dim, kappa in results.items()]
)
summary

,维度,Fleiss' Kappa
0,负面性,0.959381
1,侮辱性,0.858975


## 结果解读

- 两个维度的 $\kappa$ 均远超 0.8，按 Landis & Koch (1977) 标准属于 **almost perfect agreement**，4 位标注员的标注可作为金标准。
- 「侮辱性」的 $\kappa$（0.86）低于「负面性」（0.96），并非标注质量差异，而是**类别不平衡效应**：侮辱性正例仅约 9.5%，期望一致率 $P_e$ 被推高至 0.824，$\kappa$ 分母被压缩；两维度的观察一致率实际相近（97.99% vs 97.52%）。
- 如审稿人对不平衡敏感，可补充 Gwet's AC1（对 prevalence 更稳健）作为稳健性指标。